# Benthos Yuval EDA

Exploratory analysis of the S3-hosted Benthos Yuval dataset produced by
`verify_raw_data_and_add_to_s3.py`.

All inputs are read from S3 (`s3://dev-datamermaid-sm-sources/external_validation_datasets/benthos_yuval/`) — no local files needed.

In [ ]:
import json

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

from mermaidseg.datasets.benthos_yuval import BenthosYuvalCoralsDataset

In [ ]:
BUCKET = "dev-datamermaid-sm-sources"
PREFIX = "external_validation_datasets/benthos_yuval"

s3 = boto3.client("s3")


def s3_get_json(key: str) -> dict:
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return json.loads(body)


classes = s3_get_json(f"{PREFIX}/classes.json")
colors = s3_get_json(f"{PREFIX}/colors.json")
manifest = s3_get_json(f"{PREFIX}/manifest.json")

print(f"Num classes (incl. unlabeled): {len(classes)}")
print(f"Manifest dataset: {manifest['dataset_name']} created at {manifest['created_at_utc']}")
print(
    f"Annotations parquet: s3://{manifest['s3']['bucket']}/{manifest['s3']['annotations_parquet']}"
)

## Dataset summary (from annotations parquet)

Per-site counts are derived from the parquet on S3 (one row per (tile, unique-class)). The optional `num_uploaded_this_run` column reflects how many image+label PNG objects were written by the most recent ingestion run, sourced from `manifest['stats'][<site>]['num_uploaded_this_run']`. For pixel-level class composition see the per-site tables in `datasets/benthos_yuval/README.md`.

In [ ]:
parquet_uri = f"s3://{manifest['s3']['bucket']}/{manifest['s3']['annotations_parquet']}"
df_meta = pd.read_parquet(parquet_uri)

sites_df = (
    df_meta.groupby("site", sort=True)
    .agg(
        num_tiles=("image_id", "nunique"),
        num_label_classes=("source_label_name", "nunique"),
    )
    .rename_axis("site")
)

stats_by_site = manifest.get("stats", {})
if stats_by_site:
    sites_df["num_uploaded_this_run"] = sites_df.index.map(
        lambda s: int(stats_by_site.get(s, {}).get("num_uploaded_this_run", 0))
    )

sites_df.loc["TOTAL"] = {
    "num_tiles": df_meta["image_id"].nunique(),
    "num_label_classes": df_meta["source_label_name"].nunique(),
    **(
        {"num_uploaded_this_run": int(sites_df["num_uploaded_this_run"].sum())}
        if "num_uploaded_this_run" in sites_df.columns
        else {}
    ),
}
sites_df

## Instantiate the dataset

In [ ]:
dataset = BenthosYuvalCoralsDataset(
    source_bucket=BUCKET,
    source_s3_prefix=PREFIX,
    annotations_path=manifest["s3"]["annotations_parquet"],
)
print(f"Dataset length: {len(dataset)}")
print(f"Num source classes (incl. background): {dataset.num_source_classes}")
dataset.df_annotations.head()

## Sample tile + dense label overlay per site

Benthos Yuval is a *dense* segmentation dataset, so we draw the label as a translucent color image (one color per class from `colors.json`) rather than as point scatter.

In [ ]:
sites = sorted(dataset.df_images["site"].unique())
n = len(sites)
ncols = min(2, n) if n > 0 else 1
nrows = int(np.ceil(n / ncols)) if n > 0 else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows), squeeze=False)

rng = np.random.default_rng(0)
for ax, site in zip(axes.ravel(), sites, strict=False):
    site_rows = dataset.df_images.index[dataset.df_images["site"] == site].to_numpy()
    if len(site_rows) == 0:
        ax.axis("off")
        ax.set_title(f"{site}: no tiles")
        continue
    idx = int(rng.choice(site_rows))
    image_id = dataset.df_images.loc[idx, "image_id"]
    image = dataset.read_image(image_id=image_id, site=site)
    raw_mask = dataset.read_label(image_id=image_id, site=site)

    rgb_mask = np.zeros((*raw_mask.shape, 3), dtype=np.uint8)
    present_names = []
    for name, gid in classes.items():
        if name == "unlabeled":
            continue
        if not (raw_mask == gid).any():
            continue
        rgb = colors.get(name, [255, 0, 0])
        rgb_mask[raw_mask == gid] = np.array(rgb, dtype=np.uint8)
        present_names.append(name)

    ax.imshow(image)
    ax.imshow(rgb_mask, alpha=0.5)
    handles = [
        Patch(facecolor=np.array(colors.get(name, [255, 0, 0])) / 255.0, label=name)
        for name in sorted(present_names)[:10]
    ]
    labelled_pct = float((raw_mask != classes.get("unlabeled", 0)).mean() * 100.0)
    ax.set_title(f"{site} / {image_id}  ({labelled_pct:.1f}% labelled)")
    ax.axis("off")
    if handles:
        ax.legend(handles=handles, loc="upper right", fontsize=6, framealpha=0.7)

for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Sanity-check `__getitem__`

Confirm that `dataset[idx]` returns a `(image, source_labels)` pair where `source_labels` is a dense integer mask in the *local* source-id space (1..N, plus 0 for background), consistent with `BaseCoralDataset` semantics. The local mask is built by mapping the dense PNG (in the alphabetical `classes.json` ID space) through `dataset._classes_global_to_local`.

In [ ]:
idx = 0
image, source_mask = dataset[idx]
row = dataset.df_images.loc[idx]
print(f"Tile idx={idx}: site={row['site']}, image_id={row['image_id']}")
print(f"Image shape: {image.shape}, mask shape: {source_mask.shape}")
print(
    f"Unique local mask ids (excl. 0): {sorted({int(v) for v in np.unique(source_mask) if v != 0})[:20]}"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image)
axes[0].set_title("Image")
axes[0].axis("off")

vis = np.zeros((*source_mask.shape, 3), dtype=np.uint8)
for local_id, name in dataset.source_id2name.items():
    rgb = colors.get(name, [255, 0, 0])
    vis[source_mask == local_id] = np.array(rgb, dtype=np.uint8)

axes[1].imshow(image)
axes[1].imshow(vis, alpha=0.6)
axes[1].set_title("Source-label mask overlay (local IDs)")
axes[1].axis("off")
plt.tight_layout()
plt.show()